# LTX-2.5｜影片生成（Colab T4）

**這份是 LTX-2.5 Distilled GGUF，不是 LTX-Video 0.9.5。** 修改 `PROMPT`、`MODEL_TIER`、`MODE` 和基本參數後，依序執行、預覽並下載 MP4。 不安裝 ngrok、不啟動 Flask／FastAPI、不需對外 Web API。 **底層 ComfyUI 僅在 Colab 本機 `127.0.0.1:8188` 執行。**

T4 VRAM 約 15 GB；預設 2 秒短片、512×288、Q3_K_M，仍有 OOM 風險，必要時改 Q2_K。 第一次下載模型可能佔用大量硬碟容量及時間；建議與其他 Notebook 分開執行。

## 1. 安裝必要套件

In [ ]:
# Download ComfyUI only if it is not already installed.
# 若尚未安裝 ComfyUI，先下載程式。
!test -d /content/ComfyUI/.git || git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI

# Install the model runtime dependencies.
# 安裝模型執行環境需要的套件。
%pip -q install -r /content/ComfyUI/requirements.txt

# Install lightweight tools and compatible quantization support.
# 安裝基本工具與量化模型支援套件。
%pip -q install -U huggingface_hub hf_xet requests psutil Pillow fastapi uvicorn python-multipart "comfy-kitchen>=0.2.26"

# Install FFmpeg to create and inspect MP4 files.
# 安裝 FFmpeg，方便產生與檢查 MP4 影片。
!command -v ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)

# Download the ComfyUI extensions needed by the original LTX-2.5 workflow.
# 下載原始 LTX-2.5 工作流程所需的 ComfyUI 擴充套件。
!cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-GGUF || git clone -q --depth 1 https://github.com/city96/ComfyUI-GGUF.git)
!cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-LTXVideo || git clone -q --depth 1 https://github.com/Lightricks/ComfyUI-LTXVideo.git)
!cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-KJNodes || git clone -q --depth 1 https://github.com/kijai/ComfyUI-KJNodes.git)
!cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-Easy-Use || git clone -q --depth 1 https://github.com/yolain/ComfyUI-Easy-Use.git)
!cd /content/ComfyUI/custom_nodes && (test -d workflow-to-api-converter || git clone -q --depth 1 https://github.com/SethRobinson/comfyui-workflow-to-api-converter-endpoint.git)

# Install extension dependencies when a requirements file exists.
# 若擴充套件有相依套件清單，也一併安裝。
!for d in /content/ComfyUI/custom_nodes/*; do if [ -f "$d/requirements.txt" ]; then python -m pip install -q -r "$d/requirements.txt"; fi; done

## 2. 匯入模組

In [ ]:
# Import tools for file paths, processes, time, and HTTP requests.
# 匯入路徑、程序、時間與 HTTP 請求工具。
import os, json, sys, time, subprocess
from pathlib import Path
import requests

# Import Colab video preview and file download tools.
# 匯入 Colab 影片預覽與檔案下載工具。
from IPython.display import Video, display
from google.colab import files

# Check for an NVIDIA GPU before loading the models.
# 載入模型前先確認 NVIDIA GPU 是否可用。
import torch
assert torch.cuda.is_available(), "請先在 Colab 選擇 T4 GPU。"
print(torch.cuda.get_device_name(0))

## 3. 修改生成設定

In [ ]:
# Choose the LTX-2.5 quantization level: q3 or q2.
# 選擇 LTX-2.5 量化版本：q3 或 q2。
MODEL_TIER = "q3"

# Choose text-to-video or image-to-video.
# 選擇文字生影片或圖片生影片。
MODE = "t2v"  # "t2v" or "i2v"

# Describe the scene and its motion.
# 描述場景以及物體或鏡頭的動作。
PROMPT = "A small red boat moves slowly across a quiet lake at sunrise, gentle camera pan, cinematic"

# Start with a short, low-resolution video to reduce OOM risk.
# 先使用短秒數與較低解析度，降低 OOM 風險。
WIDTH, HEIGHT = 512, 288
SECONDS, FPS, SEED = 2.0, 24, 42

# Use an existing image only in image-to-video mode.
# 只有圖片生影片模式才需要起始圖片。
IMAGE_STRENGTH = 0.7

## 4. 下載模型與準備 Workflow（不需要修改）

In [ ]:
# Configure the original ComfyUI LTX-2.5 backend paths.
# 設定原始 LTX-2.5 ComfyUI 後端的必要路徑。
COMFY_DIR = Path("/content/ComfyUI")
FLOW_REGISTRY_PATH = Path("/content/ltx25_local_flows.json")
APP_PATH = Path("/content/app.py")
COMFY_PORT, API_PORT = 8188, 8000

# Use just one downloaded model tier to conserve disk space.
# 只下載需要的量化版本，減少儲存空間用量。
assert MODEL_TIER in {"q3", "q2"}
DOWNLOAD_Q4KM = DOWNLOAD_Q4KS = False
DOWNLOAD_Q3, DOWNLOAD_Q2 = MODEL_TIER == "q3", MODEL_TIER == "q2"
PRIMARY_MODEL_TIER, FALLBACK_MODEL_TIERS = MODEL_TIER, []

# Configure low-memory streaming and tiled video decoding.
# 使用低記憶體模型換頁及分塊影片解碼。
USE_GOOGLE_DRIVE_MODEL_CACHE = False
DRIVE_MODEL_CACHE_DIR = Path("/content/drive/MyDrive/LTX25-Colab-Cache/models")
VAE_TILE_SIZE, VAE_TILE_OVERLAP = 256, 32
VAE_TEMPORAL_SIZE, VAE_TEMPORAL_OVERLAP = 16, 4
COMFY_FAST_DISK, COMFY_DISABLE_PINNED_MEMORY = True, True
COMFY_RESERVE_VRAM_GB = 1.5

# Prevent unnecessary tokenizer parallelism and configure downloads.
# 減少額外併行程序，並設定模型下載逾時時間。
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "45")
print("LTX-2.5", MODEL_TIER, "| VRAM reserve:", COMFY_RESERVE_VRAM_GB, "GB")

In [ ]:
# Apply the small upstream compatibility adjustment if necessary.
# 如有需要，套用上游套件的相容性調整。
import sys, subprocess
from pathlib import Path

def run(cmd, cwd=None, check=True):
    return subprocess.run(cmd, cwd=cwd, check=check)

custom = COMFY_DIR / "custom_nodes"
# Current ComfyUI-LTXVideo master can still import `pad` from Kornia's pyramid
# module even though Kornia >= 0.8.3 removed that re-export. Patch only when the
# old import is present; this is idempotent.
ltx_pyramid = custom / "ComfyUI-LTXVideo" / "pyramid_blending.py"
if ltx_pyramid.is_file():
    text = ltx_pyramid.read_text(encoding="utf-8")
    changed = False
    if "    pad,\n" in text:
        text = text.replace("    pad,\n", "", 1)
        changed = True
    if changed and "pad = F.pad" not in text:
        needle = "import torch.nn.functional as F\n"
        if needle not in text:
            raise RuntimeError("Could not apply Kornia compatibility patch safely.")
        text = text.replace(needle, needle + "\n# compatibility: kornia >= 0.8.3\npad = F.pad\n", 1)
    if changed:
        ltx_pyramid.write_text(text, encoding="utf-8")
        print("Applied ComfyUI-LTXVideo Kornia pad compatibility patch.")
    else:
        print("ComfyUI-LTXVideo Kornia patch not needed/already applied.")
    run([sys.executable, "-m", "py_compile", str(ltx_pyramid)])


print("ComfyUI compatibility setup complete.")

In [ ]:
import gc
import json
import os
import shutil
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

FLOW_REGISTRY = json.loads(r'''
{
  "schema_version": 25,
  "description": "LTX-2.5 Colab T4 stack. Q3_K_M or Q2_K distilled GGUF is the selected transformer. One audited two-stage AV workflow is reused for T2V/I2V/A2V/IA2V; external audio is encoded with the existing audio VAE and injected as frozen audio latent, so no second transformer or distilled LoRA is required.",
  "models": {
    "transformers": {
      "q4km": {
        "repo_id": "realrebelai/LTX-2.5_GGUFs",
        "filename": "LTX-2.5-Distilled-Q4_K_M.gguf",
        "target_dir": "models/diffusion_models",
        "min_bytes": 11000000000
      },
      "q4ks": {
        "repo_id": "realrebelai/LTX-2.5_GGUFs",
        "filename": "LTX-2.5-Distilled-Q4_K_S.gguf",
        "target_dir": "models/diffusion_models",
        "min_bytes": 10000000000
      },
      "q3": {
        "repo_id": "realrebelai/LTX-2.5_GGUFs",
        "filename": "LTX-2.5-Distilled-Q3_K_M.gguf",
        "target_dir": "models/diffusion_models",
        "min_bytes": 8000000000
      },
      "q2": {
        "repo_id": "realrebelai/LTX-2.5_GGUFs",
        "filename": "LTX-2.5-Distilled-Q2_K.gguf",
        "target_dir": "models/diffusion_models",
        "min_bytes": 6000000000
      }
    },
    "text_encoder": {
      "repo_id": "realrebelai/Rebels_w4a8s",
      "basename": "gemma4-12b-ltx25-w4a8-v2.safetensors",
      "target_dir": "models/text_encoders",
      "min_bytes": 6000000000
    },
    "video_vae": {
      "repo_id": "comfyicu/LTX-2.5",
      "filename": "vae/ltx-2.5-video-vae-conv-bf16.safetensors",
      "target_dir": "models/vae",
      "min_bytes": 1000000000
    },
    "audio_vae": {
      "repo_id": "comfyicu/LTX-2.5",
      "filename": "vae/ltx-2.5-audio-vae-bf16.safetensors",
      "target_dir": "models/vae",
      "min_bytes": 250000000
    },
    "spatial_upscaler": {
      "repo_id": "comfyicu/LTX-2.5",
      "filename": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors",
      "target_dir": "models/latent_upscale_models",
      "min_bytes": 700000000
    }
  },
  "workflows": {
    "t2v_i2v": {
      "repo_id": "realrebelai/LTX-2.5_GGUFs",
      "revision": "112436f97aaf99ce13ecb7b7eca7e2f6c128d3ec",
      "filename": "LTX 2.5 GGUF (WORKFLOW).json",
      "local_name": "LTX25_T4_T2V_I2V.json",
      "modes": [
        "t2v",
        "i2v",
        "a2v",
        "ia2v"
      ]
    }
  },
  "notes": [
    "The chosen Q3_K_M or Q2_K transformer is downloaded by default.",
    "Spatial upscaler is required for the selected multi-stage workflow stack.",
    "A2V/IA2V reuse the same selected distilled workflow; no second transformer or distilled LoRA is downloaded.",
    "Existing audio is encoded by the LTX-2.5 audio VAE and injected as frozen audio into both sampling stages.",
    "The final A2V/IA2V file uses the original uploaded audio waveform for fidelity.",
    "Workflow converter must be >=2.3.0 for nested subgraph conversion."
  ]
}
''')
FLOW_REGISTRY_PATH.write_text(
    json.dumps(FLOW_REGISTRY, ensure_ascii=False, indent=2), encoding="utf-8"
)

if USE_GOOGLE_DRIVE_MODEL_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def target_dir(rel):
    path = COMFY_DIR / rel
    path.mkdir(parents=True, exist_ok=True)
    return path


def file_is_valid(path, min_bytes):
    path = Path(path)
    return path.is_file() and path.stat().st_size >= int(min_bytes)


def cache_path(rel_dir, target_name):
    return DRIVE_MODEL_CACHE_DIR / rel_dir / target_name


def restore_from_drive(rel_dir, target_name, min_bytes):
    if not USE_GOOGLE_DRIVE_MODEL_CACHE:
        return None
    cached = cache_path(rel_dir, target_name)
    target = target_dir(rel_dir) / target_name
    if file_is_valid(cached, min_bytes):
        print("restoring Drive cache:", cached)
        shutil.copy2(cached, target)
        if not file_is_valid(target, min_bytes):
            raise RuntimeError(f"Drive cache restore failed validation: {target}")
        return target
    return None


def save_to_drive_cache(target, rel_dir, min_bytes):
    if not USE_GOOGLE_DRIVE_MODEL_CACHE:
        return
    target = Path(target)
    cached = cache_path(rel_dir, target.name)
    cached.parent.mkdir(parents=True, exist_ok=True)
    if file_is_valid(cached, min_bytes):
        return
    print("saving Drive cache:", cached)
    shutil.copy2(target, cached)


def anonymous_download(repo_id, filename, rel_dir, min_bytes, target_name=None):
    out_dir = target_dir(rel_dir)
    target_name = target_name or Path(filename).name
    target = out_dir / target_name

    if file_is_valid(target, min_bytes):
        print("exists/validated:", target, f"({target.stat().st_size/1e9:.2f} GB)")
        return target

    if target.exists():
        print("removing invalid/truncated file:", target, target.stat().st_size)
        target.unlink()

    restored = restore_from_drive(rel_dir, target_name, min_bytes)
    if restored is not None:
        return restored

    print("downloading:", repo_id, filename)
    downloaded = Path(hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir="/content/.hf_public_stage",
    ))
    if downloaded.stat().st_size < int(min_bytes):
        raise RuntimeError(
            f"Downloaded file is unexpectedly small: {downloaded} "
            f"({downloaded.stat().st_size} bytes; expected >= {min_bytes})"
        )
    target.parent.mkdir(parents=True, exist_ok=True)
    os.replace(downloaded, target)
    gc.collect()
    subprocess.run(["sync"], check=False)
    print("saved/validated:", target, f"({target.stat().st_size/1e9:.2f} GB)")
    save_to_drive_cache(target, rel_dir, min_bytes)
    return target


def anonymous_download_by_basename(repo_id, basename, rel_dir, min_bytes):
    files = list_repo_files(repo_id=repo_id)
    matches = [f for f in files if Path(f).name == basename]
    if not matches:
        raise FileNotFoundError(f"{basename!r} not found in {repo_id}")
    filename = sorted(matches, key=lambda x: (x.count("/"), len(x)))[0]
    print("resolved basename:", basename, "->", filename)
    return anonymous_download(repo_id, filename, rel_dir, min_bytes, basename)


models = FLOW_REGISTRY["models"]
flags = {
    "q4km": DOWNLOAD_Q4KM,
    "q4ks": DOWNLOAD_Q4KS,
    "q3": DOWNLOAD_Q3,
    "q2": DOWNLOAD_Q2,
}
for tier, enabled in flags.items():
    if enabled:
        item = models["transformers"][tier]
        anonymous_download(
            item["repo_id"], item["filename"], item["target_dir"], item["min_bytes"]
        )

enc = models["text_encoder"]
anonymous_download_by_basename(
    enc["repo_id"], enc["basename"], enc["target_dir"], enc["min_bytes"]
)

# These are required by the selected workflow stack; failures must stop here.
for key in ("video_vae", "audio_vae", "spatial_upscaler"):
    item = models[key]
    anonymous_download(
        item["repo_id"], item["filename"], item["target_dir"], item["min_bytes"]
    )

# Model policy sanity check.
for tier in [PRIMARY_MODEL_TIER] + list(FALLBACK_MODEL_TIERS):
    if tier not in models["transformers"]:
        raise ValueError(f"Unknown model tier in notebook policy: {tier}")
    item = models["transformers"][tier]
    expected = COMFY_DIR / item["target_dir"] / item["filename"]
    if not file_is_valid(expected, item["min_bytes"]):
        raise RuntimeError(
            f"Configured runtime tier {tier!r} is not downloaded/valid: {expected}. "
            "Enable its DOWNLOAD_* flag and rerun this cell."
        )

print("Model downloads and size validation complete.")

In [ ]:
import json
import wave
from pathlib import Path
from huggingface_hub import hf_hub_download
from PIL import Image

WORKFLOW_DIR = COMFY_DIR / "user/default/workflows/LTX25-T4"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = COMFY_DIR / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remove stale community-audio files from older notebook versions.
for name in ("LTX25_T4_A2V.json", "LTX25_T4_IA2V.json", "LTX25_T4_AUDIO.json", "LTX25_T4_A2V_IA2V_native.json"):
    p = WORKFLOW_DIR / name
    if p.exists():
        p.unlink()

# Valid placeholders prevent unused LoadImage/LoadAudio nodes from failing validation.
PLACEHOLDER_IMAGE = "__ltx25_placeholder__.png"
PLACEHOLDER_AUDIO = "__ltx25_placeholder__.wav"
Image.new("RGB", (64, 64), (0, 0, 0)).save(INPUT_DIR / PLACEHOLDER_IMAGE, format="PNG")
with wave.open(str(INPUT_DIR / PLACEHOLDER_AUDIO), "wb") as wf:
    wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(16000)
    wf.writeframes(b"\x00\x00" * 16000)

base = FLOW_REGISTRY["workflows"]["t2v_i2v"]
src = Path(hf_hub_download(
    repo_id=base["repo_id"],
    filename=base["filename"],
    revision=base["revision"],
    local_dir="/content/.hf_workflow_stage",
))
ui = json.loads(src.read_text(encoding="utf-8"))

model_name = FLOW_REGISTRY["models"]["transformers"][PRIMARY_MODEL_TIER]["filename"]
encoder_name = FLOW_REGISTRY["models"]["text_encoder"]["basename"]
video_vae = Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name
audio_vae = Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name
upscaler = Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name


def patch_strings(obj):
    if isinstance(obj, dict):
        return {k: patch_strings(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [patch_strings(v) for v in obj]
    if not isinstance(obj, str):
        return obj
    low = obj.lower()
    if low.endswith(".gguf") and "ltx" in low:
        return model_name
    if "gemma4" in low and low.endswith(".safetensors") and "enhancer" not in low:
        return encoder_name
    if "video-vae" in low and low.endswith(".safetensors"):
        return video_vae
    if "audio-vae" in low and low.endswith(".safetensors"):
        return audio_vae
    if "upscaler" in low and low.endswith(".safetensors"):
        return upscaler
    return obj


ui = patch_strings(ui)

# Replace stale example image names with a real local placeholder.
def patch_placeholders(graph):
    for node in graph.get("nodes", []) if isinstance(graph, dict) else []:
        values = node.get("widgets_values")
        if node.get("type") == "LoadImage" and isinstance(values, list) and values:
            values[0] = PLACEHOLDER_IMAGE
        elif node.get("type") == "LoadAudio" and isinstance(values, list) and values:
            values[0] = PLACEHOLDER_AUDIO
    definitions = graph.get("definitions") if isinstance(graph, dict) else None
    for sub in (definitions or {}).get("subgraphs", []) if isinstance(definitions, dict) else []:
        patch_placeholders(sub)


patch_placeholders(ui)
out = WORKFLOW_DIR / base["local_name"]
out.write_text(json.dumps(ui, ensure_ascii=False, indent=2), encoding="utf-8")
FLOW_REGISTRY_PATH.write_text(json.dumps(FLOW_REGISTRY, ensure_ascii=False, indent=2), encoding="utf-8")
print("Workflow ready:", out)
print("Modes: T2V / I2V / A2V / IA2V")

## 5. 啟動 Colab 內部的 ComfyUI 引擎

不會開放對外網址。初始化程式較長，預設收合。

In [ ]:
# Prepare only the local ComfyUI workflow helpers, without starting a Flask/FastAPI service.
# 僅準備 Colab 內部的 ComfyUI 工作流程工具，不啟動 Flask 或 FastAPI 服務。
ENGINE_CODE = '\nimport asyncio\nimport io\nimport math\nimport re\nimport wave\nimport json\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport requests\nfrom PIL import Image\n\nCOMFY_BASE = os.environ.get("COMFY_BASE", "http://127.0.0.1:8188").rstrip("/")\nCOMFY_DIR = Path(os.environ.get("COMFY_DIR", "/content/ComfyUI"))\nFLOW_REGISTRY_PATH = Path(os.environ.get("FLOW_REGISTRY_PATH", "/content/ltx25_local_flows.json"))\nWORKFLOW_DIR = Path(os.environ.get(\n    "WORKFLOW_DIR",\n    str(COMFY_DIR / "user/default/workflows/LTX25-T4"),\n))\n\nif not FLOW_REGISTRY_PATH.is_file():\n    raise RuntimeError(f"Flow registry missing: {FLOW_REGISTRY_PATH}")\n\nFLOW_REGISTRY = json.loads(FLOW_REGISTRY_PATH.read_text(encoding="utf-8"))\n\nSTACK_SCHEMA_VERSION = int(FLOW_REGISTRY.get("schema_version", 0))\n\nMODEL_FILES = {\n    tier: item["filename"]\n    for tier, item in FLOW_REGISTRY["models"]["transformers"].items()\n}\nPRIMARY_MODEL_TIER = os.environ.get("LTX25_PRIMARY_MODEL_TIER", "q4ks").lower()\nFALLBACK_MODEL_TIERS = [\n    x.strip().lower()\n    for x in os.environ.get("LTX25_FALLBACK_MODEL_TIERS", "").split(",")\n    if x.strip()\n]\n\nVAE_TILE_SIZE = int(os.environ.get("LTX25_VAE_TILE_SIZE", "512"))\nVAE_TILE_OVERLAP = int(os.environ.get("LTX25_VAE_TILE_OVERLAP", "64"))\nVAE_TEMPORAL_SIZE = int(os.environ.get("LTX25_VAE_TEMPORAL_SIZE", "32"))\nVAE_TEMPORAL_OVERLAP = int(os.environ.get("LTX25_VAE_TEMPORAL_OVERLAP", "8"))\n\nCOMFY_LOG_PATH = Path(os.environ.get("LTX25_COMFY_LOG", "/content/comfyui_ltx25.log"))\nCOMFY_PID_PATH = Path(os.environ.get("LTX25_COMFY_PID", "/content/comfyui_ltx25.pid"))\nTEXT_ENCODER = Path(FLOW_REGISTRY["models"]["text_encoder"]["basename"]).name\nVIDEO_VAE = Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name\nAUDIO_VAE = Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name\nUPSCALER = Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name\n\nT2V_I2V_WORKFLOW = WORKFLOW_DIR / FLOW_REGISTRY["workflows"]["t2v_i2v"]["local_name"]\n\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}\nAUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".ogg"}\nMAX_IMAGE_BYTES = 15 * 1024 * 1024\nMAX_AUDIO_BYTES = 32 * 1024 * 1024\n\nJOBS = {}\nJOBS_LOCK = threading.Lock()\nEXECUTOR = ThreadPoolExecutor(max_workers=1)\n\n\n\ndef job_update(job_key, **fields):\n    # job_key is the dictionary key; fields may safely contain "job_id".\n    with JOBS_LOCK:\n        JOBS.setdefault(job_key, {}).update(fields)\n\n\ndef _comfy_http_error(method, path, response):\n    try:\n        body = response.json()\n        detail = json.dumps(body, ensure_ascii=False, indent=2)\n    except Exception:\n        detail = response.text[:12000]\n    raise RuntimeError(\n        f"ComfyUI {method} {path} failed: HTTP {response.status_code}\\n{detail}"\n    )\n\n\ndef comfy_get(path, **kwargs):\n    response = requests.get(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("GET", path, response)\n    return response\n\n\ndef comfy_post(path, **kwargs):\n    response = requests.post(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("POST", path, response)\n    return response\n\n\ndef get_gguf_loader_choices():\n    """Return (loader_ready, unet_name choices, raw node info or error)."""\n    try:\n        payload = comfy_get("/object_info/UnetLoaderGGUF", timeout=30).json()\n        node = payload.get("UnetLoaderGGUF")\n        if not isinstance(node, dict):\n            return False, [], payload\n\n        spec = (\n            node.get("input", {})\n                .get("required", {})\n                .get("unet_name")\n        )\n        choices = []\n        if isinstance(spec, (list, tuple)) and spec:\n            first = spec[0]\n            if isinstance(first, (list, tuple)):\n                choices = [str(x) for x in first]\n        return True, choices, node\n    except Exception as exc:\n        return False, [], {"error": str(exc)}\n\n\ndef free_comfy():\n    try:\n        comfy_post(\n            "/free",\n            json={"unload_models": True, "free_memory": True},\n            timeout=60,\n        )\n    except Exception:\n        pass\n\n\ndef wait_comfy_ready(timeout=120):\n    deadline = time.time() + timeout\n    last = None\n    while time.time() < deadline:\n        try:\n            response = comfy_get("/system_stats", timeout=3)\n            if response.ok:\n                return True\n        except Exception as exc:\n            last = exc\n        time.sleep(1)\n    raise RuntimeError(f"ComfyUI is not ready: {last}")\n\n\ndef validate_dimensions(width, height):\n    width = int(width)\n    height = int(height)\n    if width < 256 or height < 256:\n        raise ValueError("width and height must both be >= 256")\n    if width % 32 != 0 or height % 32 != 0:\n        raise ValueError("width and height must both be divisible by 32")\n    return width, height\n\n\ndef save_upload_bytes(data, suffix, prefix):\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = f"{prefix}_{uuid.uuid4().hex}{suffix.lower()}"\n    path = input_dir / name\n    path.write_bytes(data)\n    return name, path\n\n\ndef ensure_placeholder_image():\n    """\n    ComfyUI validates LoadImage nodes before execution even when the workflow\n    later disables image conditioning. Some downloaded UI workflows contain\n    stale example filenames such as Test__00139_.png.\n\n    Keep one valid tiny placeholder in ComfyUI/input and patch unused LoadImage\n    nodes to it. The workflow\'s use-image switch still controls conditioning.\n    """\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n\n    name = "__ltx25_placeholder__.png"\n    path = input_dir / name\n\n    if not path.is_file():\n        Image.new("RGB", (64, 64), (0, 0, 0)).save(path, format="PNG")\n\n    return name\n\n\ndef ensure_placeholder_audio():\n    """Create a valid one-second silent WAV for unused LoadAudio validation."""\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = "__ltx25_placeholder__.wav"\n    path = input_dir / name\n    if not path.is_file():\n        with wave.open(str(path), "wb") as wf:\n            wf.setnchannels(1)\n            wf.setsampwidth(2)\n            wf.setframerate(16000)\n            wf.writeframes(b"\\x00\\x00" * 16000)\n    return name\n\n\ndef validate_image_bytes(data, suffix):\n    if not data:\n        raise ValueError("Image is empty.")\n    if len(data) > MAX_IMAGE_BYTES:\n        raise ValueError(f"Image exceeds {MAX_IMAGE_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in IMAGE_EXTENSIONS:\n        raise ValueError(f"Unsupported image extension: {suffix}")\n    try:\n        with Image.open(io.BytesIO(data)) as image:\n            image.verify()\n    except Exception as exc:\n        raise ValueError("Invalid/corrupted image.") from exc\n\n\ndef ffprobe_audio_bytes(data, suffix):\n    if not data:\n        raise ValueError("Audio is empty.")\n    if len(data) > MAX_AUDIO_BYTES:\n        raise ValueError(f"Audio exceeds {MAX_AUDIO_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in AUDIO_EXTENSIONS:\n        raise ValueError(f"Unsupported audio extension: {suffix}")\n\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:\n        tmp.write(data)\n        path = tmp.name\n\n    try:\n        result = subprocess.run(\n            [\n                "ffprobe", "-v", "error",\n                "-select_streams", "a:0",\n                "-show_entries", "stream=codec_name,profile:format=duration",\n                "-of", "json",\n                path,\n            ],\n            capture_output=True,\n            text=True,\n            timeout=30,\n        )\n        if result.returncode != 0:\n            raise ValueError("ffprobe could not decode the audio.")\n        payload = json.loads(result.stdout or "{}")\n        streams = payload.get("streams") or []\n        if not streams:\n            raise ValueError("No audio stream found.")\n        duration = float((payload.get("format") or {}).get("duration") or 0)\n        if duration <= 0:\n            raise ValueError("Could not determine audio duration.")\n        return {\n            "codec": streams[0].get("codec_name"),\n            "profile": streams[0].get("profile"),\n            "duration": duration,\n        }\n    finally:\n        try:\n            os.remove(path)\n        except OSError:\n            pass\n\n\ndef _norm_label(value):\n    text = str(value or "").strip().lower().replace("_", " ")\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\nALIASES = {\n    "prompt": {"prompt", "positive prompt", "prompt positive", "text prompt", "prompt text", "main prompt", "positive text"},\n    "width": {"width", "video width", "output width"},\n    "height": {"height", "video height", "output height"},\n    "seconds": {"duration", "duration in seconds", "seconds", "video duration"},\n    "fps": {"fps", "frame rate", "frames per second"},\n    "seed": {"seed", "noise seed"},\n    "image_strength": {"img strength", "image strength"},\n    "video_vae": {"video vae"},\n    "audio_vae": {"audio vae"},\n    "text_encoder": {"text encoder", "encoder name", "clip name"},\n    "upscaler": {"upscale model", "upscaler", "spatial upscaler"},\n    "text_to_video": {"text to video", "t2v"},\n    "use_image": {"use image input", "use image", "image input"},\n    "use_audio": {"use custom audio", "custom audio", "use audio", "audio input"},\n    "prompt_enhance": {"enhance positive prompt", "prompt enhance", "enhance prompt"},\n    "tile_size": {"tile size"},\n    "tile_overlap": {"tile overlap"},\n    "temporal_size": {"temporal size"},\n    "temporal_overlap": {"temporal overlap"},\n    "model": {"ltx model", "diffusion model", "unet", "unet name", "model checkpoint"},\n}\nALIASES = {k: {_norm_label(x) for x in v} for k, v in ALIASES.items()}\n\n\ndef _matches_field_label(field, label):\n    label = _norm_label(label)\n    if label in ALIASES[field]:\n        return True\n    if field == "prompt" and "prompt" in label:\n        blocked = ("negative", "enhanc", "system", "instruction", "translated", "output")\n        return not any(word in label for word in blocked)\n    return False\n\n\ndef _iter_all_nodes(ui):\n    if not isinstance(ui, dict):\n        return\n    for node in ui.get("nodes", []):\n        if isinstance(node, dict):\n            yield ui, node\n    definitions = ui.get("definitions")\n    subgraphs = definitions.get("subgraphs", []) if isinstance(definitions, dict) else []\n    for subgraph in subgraphs:\n        if isinstance(subgraph, dict):\n            yield from _iter_all_nodes(subgraph)\n\n\ndef _remove_ui_link(owner, link_id):\n    links = owner.get("links")\n    if not isinstance(links, list) or link_id is None:\n        return\n    kept = []\n    for link in links:\n        if isinstance(link, list) and link and link[0] == link_id:\n            continue\n        if isinstance(link, dict) and link.get("id") == link_id:\n            continue\n        kept.append(link)\n    owner["links"] = kept\n\n\ndef _widget_inputs(node):\n    return [x for x in (node.get("inputs") or []) if isinstance(x, dict) and "widget" in x]\n\n\ndef _widget_index(node, input_name):\n    for i, inp in enumerate(_widget_inputs(node)):\n        if inp.get("name") == input_name:\n            return i\n    return None\n\n\ndef _coerce_widget_value(field, value, ui_type):\n    ui_type = str(ui_type or "").upper()\n    if ui_type == "BOOLEAN":\n        return bool(value)\n    if ui_type == "INT":\n        if field == "seconds":\n            return max(1, int(math.ceil(float(value))))\n        return int(round(float(value)))\n    if ui_type == "FLOAT":\n        return float(value)\n    if ui_type in {"STRING", "COMBO"}:\n        return str(value)\n    return value\n\n\ndef _set_exact_widget(owner, node, field, value, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list):\n        return\n    for inp in node.get("inputs") or []:\n        if not isinstance(inp, dict):\n            continue\n        label = _norm_label(inp.get("label") or inp.get("name"))\n        if not _matches_field_label(field, label):\n            continue\n        idx = _widget_index(node, inp.get("name"))\n        if idx is None or idx >= len(values):\n            continue\n        # User-requested controls can be linked to helper nodes in a UI workflow\n        # (the current GGUF workflow links width/height to ResolutionSelector).\n        # Disconnect that incoming UI link so the patched widget value is the\n        # value that survives UI -> API conversion.\n        if field in {\n            "prompt", "width", "height", "seconds", "fps", "seed",\n            "image_strength", "text_to_video", "use_image", "use_audio",\n            "prompt_enhance", "tile_size", "tile_overlap",\n            "temporal_size", "temporal_overlap",\n        } and inp.get("link") is not None:\n            link_id = inp.get("link")\n            _remove_ui_link(owner, link_id)\n            inp["link"] = None\n            stats[f"disconnected_{field}"] = stats.get(f"disconnected_{field}", 0) + 1\n        values[idx] = _coerce_widget_value(field, value, inp.get("type"))\n        stats[field] = stats.get(field, 0) + 1\n\n\ndef _patch_primitives(node, values_by_field, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list) or not values:\n        return\n    node_type = str(node.get("type") or "")\n    title = _norm_label(node.get("title"))\n    if node_type in {"PrimitiveString", "PrimitiveStringMultiline"} and _matches_field_label("prompt", title):\n        values[0] = values_by_field["prompt"]\n        stats["prompt"] = stats.get("prompt", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["seconds"]:\n        values[0] = int(math.ceil(values_by_field["seconds"])) if node_type == "PrimitiveInt" else float(values_by_field["seconds"])\n        stats["seconds"] = stats.get("seconds", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["fps"]:\n        values[0] = int(values_by_field["fps"]) if node_type == "PrimitiveInt" else float(values_by_field["fps"])\n        stats["fps"] = stats.get("fps", 0) + 1\n    elif node_type == "PrimitiveInt" and title in ALIASES["seed"]:\n        values[0] = int(values_by_field["seed"])\n        stats["seed"] = stats.get("seed", 0) + 1\n\n\ndef patch_workflow(\n    ui, *, mode, prompt, image_name, audio_name, model_file,\n    width, height, seconds, fps, seed, image_strength,\n    tile_size, tile_overlap, temporal_size, temporal_overlap,\n):\n    text_mode = mode in {"t2v", "a2v"}\n    use_image = mode in {"i2v", "ia2v"}\n    use_audio = mode in {"a2v", "ia2v"}\n    values_by_field = {\n        "prompt": prompt, "width": width, "height": height, "seconds": seconds,\n        "fps": fps, "seed": seed, "image_strength": image_strength,\n        "video_vae": VIDEO_VAE, "audio_vae": AUDIO_VAE,\n        "text_encoder": TEXT_ENCODER, "upscaler": UPSCALER,\n        "text_to_video": text_mode, "use_image": use_image, "use_audio": use_audio,\n        "prompt_enhance": False, "tile_size": tile_size, "tile_overlap": tile_overlap,\n        "temporal_size": temporal_size, "temporal_overlap": temporal_overlap,\n        "model": model_file,\n    }\n    stats = {}\n\n    for owner, node in _iter_all_nodes(ui):\n        node_type = str(node.get("type") or "")\n        widgets = node.get("widgets_values")\n        _patch_primitives(node, values_by_field, stats)\n\n        if node_type == "LoadImage" and isinstance(widgets, list) and widgets:\n            widgets[0] = image_name\n            stats["load_image"] = stats.get("load_image", 0) + 1\n        elif node_type == "LoadAudio" and isinstance(widgets, list) and widgets:\n            widgets[0] = audio_name\n            stats["load_audio"] = stats.get("load_audio", 0) + 1\n        elif node_type in {"UnetLoaderGGUF", "UNETLoader"} and isinstance(widgets, list) and widgets:\n            widgets[0] = model_file\n            stats["model"] = stats.get("model", 0) + 1\n        elif node_type == "LatentUpscaleModelLoader" and isinstance(widgets, list) and widgets:\n            widgets[0] = UPSCALER\n            stats["upscaler"] = stats.get("upscaler", 0) + 1\n\n        for field, value in values_by_field.items():\n            _set_exact_widget(owner, node, field, value, stats)\n\n    # Contract checks: fail here rather than silently generating with stale defaults.\n    required = ["prompt", "width", "height", "seconds", "fps", "seed", "model"]\n    if mode in {"i2v", "ia2v"}:\n        required.append("load_image")\n    missing = [name for name in required if stats.get(name, 0) == 0]\n    if missing:\n        raise RuntimeError(\n            f"Workflow patch contract failed for mode={mode}: missing controls/nodes {missing}. "\n            f"Patch stats={stats}"\n        )\n    return ui, stats\n\n\n_OBJECT_INFO_CACHE = {"at": 0.0, "data": None}\n\n\ndef get_object_info_all(max_age=60):\n    now = time.time()\n    if _OBJECT_INFO_CACHE["data"] is None or now - _OBJECT_INFO_CACHE["at"] > max_age:\n        _OBJECT_INFO_CACHE["data"] = comfy_get("/object_info", timeout=60).json()\n        _OBJECT_INFO_CACHE["at"] = now\n    return _OBJECT_INFO_CACHE["data"]\n\n\ndef convert_workflow(ui_workflow):\n    response = comfy_post("/workflow/convert", json=ui_workflow, timeout=120)\n    payload = response.json()\n    if not isinstance(payload, dict) or not any(\n        isinstance(v, dict) and "class_type" in v for v in payload.values()\n    ):\n        raise RuntimeError(f"Workflow converter returned unexpected payload: {payload}")\n    return payload\n\n\ndef normalize_nested_random_noise(api_workflow):\n    """Use local Gaussian noise that preserves nested video/audio structure."""\n    converted = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "RandomNoise":\n            node["class_type"] = "LTX25NestedRandomNoise"\n            converted += 1\n    if converted == 0:\n        raise RuntimeError("Converted workflow contains no RandomNoise nodes to normalize.")\n    return converted\n\n\ndef normalize_api_workflow_for_gguf(api_workflow, model_file):\n    """\n    Defensive API-side normalization for unusual nested groups that still\n    convert to ordinary UNETLoader.\n    """\n    converted = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        class_type = str(node.get("class_type") or "")\n        inputs = node.get("inputs")\n        if not isinstance(inputs, dict):\n            inputs = {}\n            node["inputs"] = inputs\n\n        if class_type == "UNETLoader":\n            node["class_type"] = "UnetLoaderGGUF"\n            node["inputs"] = {"unet_name": model_file}\n            converted += 1\n        elif class_type == "UnetLoaderGGUF":\n            inputs["unet_name"] = model_file\n    return converted\n\n\ndef inject_frozen_external_audio(api_workflow, audio_name, seconds):\n    """Inject existing audio into both stages of the distilled AV workflow.\n\n    Local-only path:\n      LoadAudio -> TrimAudioDuration -> VAEEncodeAudio\n      -> LTXVSetAudioRefTokens -> frozen_audio\n      -> every active LTXVConcatAVLatent audio input\n\n    The normal positive/negative conditioning graph is left unchanged; only the\n    node\'s frozen_audio output is used. For audio-driven modes, both samplers are\n    switched to plain Euler.\n    """\n    if not audio_name:\n        raise RuntimeError("External audio injection requires audio_name.")\n\n    audio_vae_loader = None\n    conditioning_node = None\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "VAELoader" and (node.get("inputs") or {}).get("vae_name") == AUDIO_VAE:\n            audio_vae_loader = str(node_id)\n        if conditioning_node is None and node.get("class_type") == "LTXVConditioning":\n            conditioning_node = str(node_id)\n\n    if audio_vae_loader is None:\n        raise RuntimeError(f"Audio VAELoader not found for {AUDIO_VAE!r}.")\n    if conditioning_node is None:\n        raise RuntimeError("LTXVConditioning not found in converted workflow.")\n\n    def fresh_id(base):\n        candidate = base\n        n = 1\n        while candidate in api_workflow:\n            n += 1\n            candidate = f"{base}_{n}"\n        return candidate\n\n    load_id = fresh_id("ltx25_external_audio")\n    trim_id = fresh_id("ltx25_trim_audio")\n    encode_id = fresh_id("ltx25_encode_audio")\n    freeze_id = fresh_id("ltx25_freeze_audio")\n\n    api_workflow[load_id] = {\n        "class_type": "LoadAudio",\n        "inputs": {"audio": audio_name},\n        "_meta": {"title": "LTX25 External Audio"},\n    }\n    api_workflow[trim_id] = {\n        "class_type": "TrimAudioDuration",\n        "inputs": {\n            "audio": [load_id, 0],\n            "start_index": 0.0,\n            "duration": float(seconds),\n        },\n        "_meta": {"title": "LTX25 Trim External Audio"},\n    }\n    api_workflow[encode_id] = {\n        "class_type": "VAEEncodeAudio",\n        "inputs": {\n            "audio": [trim_id, 0],\n            "vae": [audio_vae_loader, 0],\n        },\n        "_meta": {"title": "LTX25 Encode External Audio"},\n    }\n    api_workflow[freeze_id] = {\n        "class_type": "LTXVSetAudioRefTokens",\n        "inputs": {\n            "positive": [conditioning_node, 0],\n            "negative": [conditioning_node, 1],\n            "audio_latent": [encode_id, 0],\n        },\n        "_meta": {"title": "LTX25 Freeze External Audio"},\n    }\n\n    concat_count = 0\n    sampler_count = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        class_type = node.get("class_type")\n        inputs = node.get("inputs")\n        if not isinstance(inputs, dict):\n            continue\n        if class_type == "LTXVConcatAVLatent":\n            # Official distilled A2V keeps the same encoded audio frozen in both stages.\n            inputs["audio_latent"] = [freeze_id, 2]\n            concat_count += 1\n        elif class_type == "KSamplerSelect":\n            inputs["sampler_name"] = "euler"\n            sampler_count += 1\n\n    if concat_count < 2:\n        raise RuntimeError(\n            "Expected a two-stage AV workflow with at least two LTXVConcatAVLatent nodes; "\n            f"found {concat_count}. The pinned workflow may have changed."\n        )\n    if sampler_count < 2:\n        raise RuntimeError(\n            "Expected at least two KSamplerSelect nodes for the two-stage audio path; "\n            f"found {sampler_count}."\n        )\n\n    return {\n        "frozen_audio_concat_nodes": concat_count,\n        "euler_samplers": sampler_count,\n    }\n\n\ndef _is_connection(value):\n    return (\n        isinstance(value, list) and len(value) == 2\n        and isinstance(value[0], (str, int)) and isinstance(value[1], int)\n    )\n\n\ndef validate_api_workflow(api_workflow):\n    """Structural validation against ComfyUI /object_info without queueing GPU work.\\n\\n    Supports normal required inputs plus newer grouped/dynamic inputs\\n    (for example Autogrow values.a / values.b / ...).\\n    """\n    object_info = get_object_info_all()\n    errors = []\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict) or "class_type" not in node:\n            continue\n        class_type = node["class_type"]\n        spec = object_info.get(class_type)\n        if not isinstance(spec, dict):\n            errors.append(f"node {node_id}: class_type {class_type!r} is not registered")\n            continue\n        inputs = node.get("inputs") or {}\n        groups = spec.get("input") or {}\n        required = groups.get("required") or {}\n        optional = groups.get("optional") or {}\n        for name in required:\n            if name in inputs:\n                continue\n\n            # Switch branches are lazy/conditional inputs. Only the branch\n            # selected by `switch` is requested by current ComfyUI.\n            if class_type in {"ComfySwitchNode", "ComfySoftSwitchNode"} and name in {"on_false", "on_true"}:\n                switch_value = inputs.get("switch")\n                if isinstance(switch_value, bool):\n                    selected = "on_true" if switch_value else "on_false"\n                    if name != selected:\n                        continue\n                    # If the statically selected branch is missing, fall through\n                    # and report it. A linked/dynamic switch is deferred.\n                else:\n                    continue\n\n            # ComfyUI\'s newer schema supports grouped/dynamic inputs such as\n            # io.Autogrow.Input("values", ...). The converted API workflow does\n            # not contain a literal "values" key; it contains children such as\n            # "values.a", "values.b", ... instead.\n            #\n            # Treat the required group as present when at least one child key\n            # exists. This is generic and is not hard-coded to ComfyMathExpression.\n            dynamic_prefix = f"{name}."\n            has_dynamic_child = any(\n                isinstance(input_name, str) and input_name.startswith(dynamic_prefix)\n                for input_name in inputs\n            )\n            if has_dynamic_child:\n                continue\n\n            errors.append(f"node {node_id} {class_type}: missing required input {name!r}")\n\n        definitions = dict(optional)\n        definitions.update(required)\n        for name, value in inputs.items():\n            # LoadImage/LoadAudio file lists are dynamic. /object_info may have\n            # been cached before a request uploaded its file, so do not use the\n            # cached combo list as our own file-existence validator. ComfyUI\'s\n            # node validation remains authoritative when /prompt is queued.\n            if (class_type == "LoadAudio" and name == "audio") or (class_type == "LoadImage" and name == "image"):\n                continue\n\n            definition = definitions.get(name)\n            if not isinstance(definition, (list, tuple)) or not definition:\n                continue\n            choices = definition[0]\n            if isinstance(choices, list) and not _is_connection(value):\n                if value not in choices:\n                    preview = choices[:20]\n                    errors.append(\n                        f"node {node_id} {class_type}.{name}: {value!r} not in selectable choices {preview!r}"\n                    )\n    if errors:\n        raise RuntimeError("ComfyUI structural preflight failed:\\n" + "\\n".join(errors[:50]))\n    return True\n\n\ndef queue_prompt(api_workflow):\n    response = comfy_post(\n        "/prompt",\n        json={"prompt": api_workflow, "client_id": str(uuid.uuid4())},\n        timeout=120,\n    )\n    payload = response.json()\n    if payload.get("node_errors"):\n        raise RuntimeError(\n            "ComfyUI rejected workflow:\\n"\n            + json.dumps(payload["node_errors"], ensure_ascii=False, indent=2)\n        )\n    prompt_id = payload.get("prompt_id")\n    if not prompt_id:\n        raise RuntimeError(f"ComfyUI did not return prompt_id: {payload}")\n    return prompt_id\n\n\ndef comfy_crash_diagnostics():\n    parts = []\n\n    pid = None\n    try:\n        if COMFY_PID_PATH.is_file():\n            pid = int(COMFY_PID_PATH.read_text(encoding="utf-8").strip())\n            try:\n                os.kill(pid, 0)\n                parts.append(f"ComfyUI PID {pid} still exists.")\n            except OSError:\n                parts.append(f"ComfyUI PID {pid} is no longer running.")\n    except Exception as exc:\n        parts.append(f"PID check failed: {exc}")\n\n    try:\n        meminfo = {}\n        for line in Path("/proc/meminfo").read_text().splitlines():\n            if ":" in line:\n                key, value = line.split(":", 1)\n                meminfo[key] = value.strip()\n        parts.append(\n            "Host memory after failure: "\n            f"MemAvailable={meminfo.get(\'MemAvailable\', \'?\')}, "\n            f"SwapFree={meminfo.get(\'SwapFree\', \'?\')}."\n        )\n    except Exception:\n        pass\n\n    try:\n        gpu = subprocess.run(\n            [\n                "nvidia-smi",\n                "--query-gpu=memory.total,memory.used,memory.free",\n                "--format=csv,noheader,nounits",\n            ],\n            capture_output=True,\n            text=True,\n            timeout=15,\n        )\n        if gpu.stdout.strip():\n            parts.append("GPU MiB total,used,free: " + gpu.stdout.strip())\n    except Exception:\n        pass\n\n    try:\n        if COMFY_LOG_PATH.is_file():\n            tail = COMFY_LOG_PATH.read_text(errors="replace")[-7000:]\n            parts.append("ComfyUI log tail:\\n" + tail)\n    except Exception as exc:\n        parts.append(f"Could not read ComfyUI log: {exc}")\n\n    return "\\n".join(parts)\n\n\ndef _looks_like_connection_loss(exc):\n    text = str(exc).lower()\n    return (\n        "connection refused" in text\n        or "failed to establish a new connection" in text\n        or "max retries exceeded" in text\n    )\n\n\ndef wait_prompt(prompt_id, timeout=3600):\n    deadline = time.time() + timeout\n    while time.time() < deadline:\n        try:\n            payload = comfy_get(\n                f"/history/{prompt_id}",\n                timeout=60,\n            ).json()\n        except Exception as exc:\n            if _looks_like_connection_loss(exc):\n                raise RuntimeError(\n                    "ComfyUI became unreachable during generation. "\n                    "The ComfyUI process probably exited or was killed by memory pressure.\\n"\n                    + comfy_crash_diagnostics()\n                ) from exc\n            raise\n\n        if prompt_id in payload:\n            history = payload[prompt_id]\n            status = history.get("status", {})\n            if status.get("status_str") == "error":\n                raise RuntimeError(\n                    json.dumps(status.get("messages", []), ensure_ascii=False, indent=2)\n                )\n            return history\n        time.sleep(3)\n    raise TimeoutError(f"Timed out waiting for ComfyUI prompt {prompt_id}")\n\n\ndef extract_output_item(history):\n    # Prefer video-like outputs. Some nodes use "videos", older wrappers use\n    # "gifs", and newer/custom nodes may expose generic "files".\n    video_exts = {".mp4", ".webm", ".mov", ".mkv", ".gif"}\n    fallback = None\n\n    for output in history.get("outputs", {}).values():\n        if not isinstance(output, dict):\n            continue\n        for key in ("videos", "gifs", "images", "files"):\n            items = output.get(key, [])\n            if not isinstance(items, list):\n                continue\n            for item in items:\n                if not isinstance(item, dict) or not item.get("filename"):\n                    continue\n                suffix = Path(item["filename"]).suffix.lower()\n                if key in {"videos", "gifs"} or suffix in video_exts:\n                    return item\n                if fallback is None:\n                    fallback = item\n\n    if fallback is not None:\n        return fallback\n    raise RuntimeError("Generation completed but no output file was found in ComfyUI history.")\n\n\ndef materialize_output(job_id, item):\n    output_dir = Path("/content/ltx25_api_outputs")\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    filename = item.get("filename", f"{job_id}.mp4")\n    suffix = Path(filename).suffix or ".mp4"\n    target = output_dir / f"{job_id}{suffix}"\n\n    response = comfy_get(\n        "/view",\n        params=item,\n        timeout=600,\n    )\n    target.write_bytes(response.content)\n    return target\n\n\ndef mux_original_audio(job_id, video_path, audio_name):\n    """Mux the original uploaded audio back onto the generated video.\n\n    The latent audio drives generation, but the delivered MP4 uses the user\'s\n    original waveform rather than a VAE-decoded reconstruction.\n    """\n    audio_path = COMFY_DIR / "input" / audio_name\n    if not audio_path.is_file():\n        raise RuntimeError(f"Uploaded audio disappeared: {audio_path}")\n\n    output_dir = Path("/content/ltx25_api_outputs")\n    target = output_dir / f"{job_id}_with_input_audio.mp4"\n    cmd = [\n        "ffmpeg", "-y", "-v", "error",\n        "-i", str(video_path),\n        "-i", str(audio_path),\n        "-map", "0:v:0",\n        "-map", "1:a:0",\n        "-c:v", "copy",\n        "-c:a", "aac",\n        "-b:a", "192k",\n        "-shortest",\n        str(target),\n    ]\n    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)\n    if result.returncode != 0:\n        raise RuntimeError(f"ffmpeg audio mux failed: {result.stderr[:4000]}")\n    return target\n\n\ndef looks_like_oom(exc):\n    text = str(exc).lower()\n    needles = [\n        "cuda out of memory",\n        "out of memory",\n        "not enough memory",\n        "failed to allocate",\n        "cublas_status_alloc_failed",\n        "cudaerror: out of memory",\n    ]\n    return any(n in text for n in needles)\n\n\ndef choose_workflow(mode):\n    if mode not in {"t2v", "i2v", "a2v", "ia2v"}:\n        raise RuntimeError(f"Unsupported mode: {mode}")\n    if not T2V_I2V_WORKFLOW.is_file():\n        raise RuntimeError(f"Base workflow missing: {T2V_I2V_WORKFLOW}")\n    return T2V_I2V_WORKFLOW\n\n\ndef prepare_api_workflow(params, tier):\n    mode = params["mode"]\n    model_file = MODEL_FILES[tier]\n    workflow_path = choose_workflow(mode)\n    ui = json.loads(workflow_path.read_text(encoding="utf-8"))\n    ui, patch_stats = patch_workflow(\n        ui,\n        mode=mode,\n        prompt=params["prompt"],\n        image_name=params["image_name"],\n        audio_name=params["audio_name"],\n        model_file=model_file,\n        width=params["width"],\n        height=params["height"],\n        seconds=params["seconds"],\n        fps=params["fps"],\n        seed=params["seed"],\n        image_strength=params["image_strength"],\n        tile_size=params["vae_tile_size"],\n        tile_overlap=params["vae_tile_overlap"],\n        temporal_size=params["vae_temporal_size"],\n        temporal_overlap=params["vae_temporal_overlap"],\n    )\n    api_workflow = convert_workflow(ui)\n    nested_noise_nodes = normalize_nested_random_noise(api_workflow)\n    patch_stats["nested_random_noise"] = nested_noise_nodes\n    normalized_loaders = normalize_api_workflow_for_gguf(api_workflow, model_file)\n    if normalized_loaders:\n        patch_stats["api_unet_to_gguf"] = normalized_loaders\n    if mode in {"a2v", "ia2v"}:\n        audio_stats = inject_frozen_external_audio(\n            api_workflow, params["audio_name"], params["seconds"]\n        )\n        patch_stats.update(audio_stats)\n    validate_api_workflow(api_workflow)\n    return workflow_path, api_workflow, patch_stats\n\n\ndef run_once(job_id, params, tier):\n    model_file = MODEL_FILES[tier]\n\n    loader_ready, gguf_choices, _ = get_gguf_loader_choices()\n    if not loader_ready:\n        raise RuntimeError("ComfyUI node UnetLoaderGGUF is not registered.")\n    if model_file not in gguf_choices:\n        raise RuntimeError(\n            f"Model exists on disk but UnetLoaderGGUF cannot select {model_file!r}. "\n            f"Choices={gguf_choices}"\n        )\n\n    job_update(job_id, status="preparing", message=f"Patching/validating workflow with {tier}.")\n    workflow_path, api_workflow, patch_stats = prepare_api_workflow(params, tier)\n    job_update(\n        job_id,\n        workflow=str(workflow_path),\n        patch_stats=patch_stats,\n        message=f"Workflow structurally validated with {tier}; queueing ComfyUI.",\n    )\n\n    free_comfy()\n    time.sleep(1)\n    prompt_id = queue_prompt(api_workflow)\n    job_update(\n        job_id,\n        prompt_id=prompt_id,\n        status="running",\n        message=f"ComfyUI is generating with {tier}.",\n    )\n\n    history = wait_prompt(prompt_id)\n    item = extract_output_item(history)\n    result_path = materialize_output(job_id, item)\n    if params["mode"] in {"a2v", "ia2v"}:\n        result_path = mux_original_audio(job_id, result_path, params["audio_name"])\n    free_comfy()\n    return result_path\n\n\ndef worker(job_id, params):\n    try:\n        job_update(job_id, status="preparing", message="Preparing local ComfyUI workflow.")\n        # Model policy comes ONLY from the Colab notebook environment.\n        configured_chain = [PRIMARY_MODEL_TIER] + [\n            tier for tier in FALLBACK_MODEL_TIERS\n            if tier != PRIMARY_MODEL_TIER\n        ]\n\n        for tier in configured_chain:\n            if tier not in MODEL_FILES:\n                raise RuntimeError(\n                    f"Notebook model policy contains unsupported tier {tier!r}. "\n                    f"Supported: {sorted(MODEL_FILES)}"\n                )\n\n        # Skip configured fallback tiers that have not been downloaded.\n        available_chain = []\n        for tier in configured_chain:\n            model_path = COMFY_DIR / "models/diffusion_models" / MODEL_FILES[tier]\n            if model_path.is_file():\n                available_chain.append(tier)\n\n        if not available_chain:\n            raise RuntimeError(\n                "No configured model tier is downloaded. "\n                f"Notebook policy: {configured_chain}"\n            )\n\n        result = None\n        last_error = None\n        for index, tier in enumerate(available_chain):\n            try:\n                result = run_once(job_id, params, tier)\n                break\n            except Exception as exc:\n                last_error = exc\n                is_last = index == len(available_chain) - 1\n                if is_last or not looks_like_oom(exc):\n                    raise\n\n                next_tier = available_chain[index + 1]\n                free_comfy()\n                job_update(\n                    job_id,\n                    status="retrying",\n                    active_model_tier=next_tier,\n                    message=f"{tier} failed with an OOM-like error; retrying {next_tier}.",\n                    last_tier_error=str(exc),\n                )\n\n        if result is None:\n            raise RuntimeError(f"All model tiers failed. Last error: {last_error}")\n\n        job_update(\n            job_id,\n            status="completed",\n            result_path=str(result),\n            message="Generation completed.",\n        )\n\n    except Exception as exc:\n        free_comfy()\n        job_update(\n            job_id,\n            status="failed",\n            error=str(exc),\n            message="Generation failed.",\n        )\n\n\ndef _sample_preflight_params(mode):\n    return {\n        "mode": mode,\n        "prompt": "LTX-2.5 structural preflight",\n        "width": 608,\n        "height": 352,\n        "seconds": 3.0,\n        "fps": 24,\n        "seed": 42,\n        "image_strength": 0.7,\n        "vae_tile_size": VAE_TILE_SIZE,\n        "vae_tile_overlap": VAE_TILE_OVERLAP,\n        "vae_temporal_size": VAE_TEMPORAL_SIZE,\n        "vae_temporal_overlap": VAE_TEMPORAL_OVERLAP,\n        "image_name": ensure_placeholder_image(),\n        "audio_name": ensure_placeholder_audio(),\n        "audio_info": None,\n    }\n\n\n_PREFLIGHT_CACHE = {"at": 0.0, "data": None}\n\n\ndef preflight_all_modes(max_age=30):\n    now = time.time()\n    if _PREFLIGHT_CACHE["data"] is not None and now - _PREFLIGHT_CACHE["at"] <= max_age:\n        return _PREFLIGHT_CACHE["data"]\n    result = {}\n    for mode in ("t2v", "i2v", "a2v", "ia2v"):\n        try:\n            workflow_path, _, stats = prepare_api_workflow(\n                _sample_preflight_params(mode), PRIMARY_MODEL_TIER\n            )\n            result[mode] = {\n                "ok": True,\n                "workflow": str(workflow_path),\n                "patch_stats": stats,\n            }\n        except Exception as exc:\n            result[mode] = {\n                "ok": False,\n                "workflow": str(T2V_I2V_WORKFLOW),\n                "error": str(exc),\n            }\n    _PREFLIGHT_CACHE["at"] = now\n    _PREFLIGHT_CACHE["data"] = result\n    return result\n\n\n'
Path("/content/ltx25_engine.py").write_text(ENGINE_CODE, encoding="utf-8")
print("Local LTX-2.5 engine prepared (no Web API server).")

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path
import requests

COMFY_LOG = Path("/content/comfyui_ltx25.log")

# ------------------------------------------------------------------
# LTX-2.5 local nested-noise node.
# No ComfyUI core patch: generate Gaussian noise per NestedTensor component.
# ------------------------------------------------------------------
LTX25_NOISE_DIR = COMFY_DIR / "custom_nodes" / "LTX25LocalNoise"
LTX25_NOISE_DIR.mkdir(parents=True, exist_ok=True)
LTX25_NOISE_CODE = r'''import numpy as np
import torch
import comfy.nested_tensor


def _noise_for_tensor(tensor, generator, noise_inds=None):
    if noise_inds is None:
        return torch.randn(
            tensor.size(), dtype=torch.float32, layout=tensor.layout,
            generator=generator, device="cpu"
        ).to(dtype=tensor.dtype)

    unique_inds, inverse = np.unique(noise_inds, return_inverse=True)
    noises = []
    for i in range(unique_inds[-1] + 1):
        noise = torch.randn(
            [1] + list(tensor.size())[1:], dtype=torch.float32,
            layout=tensor.layout, generator=generator, device="cpu"
        ).to(dtype=tensor.dtype)
        if i in unique_inds:
            noises.append(noise)
    noises = [noises[i] for i in inverse]
    return torch.cat(noises, axis=0)


class _LTX25NestedNoise:
    def __init__(self, seed):
        self.seed = int(seed)

    def generate_noise(self, input_latent):
        latent = input_latent["samples"]
        noise_inds = input_latent.get("batch_index")
        generator = torch.manual_seed(self.seed)

        if getattr(latent, "is_nested", False):
            parts = [
                _noise_for_tensor(part, generator, noise_inds)
                for part in latent.unbind()
            ]
            return comfy.nested_tensor.NestedTensor(parts)

        return _noise_for_tensor(latent, generator, noise_inds)


class LTX25NestedRandomNoise:
    @classmethod
    def INPUT_TYPES(cls):
        return {
            "required": {
                "noise_seed": (
                    "INT",
                    {
                        "default": 0,
                        "min": 0,
                        "max": 0xFFFFFFFFFFFFFFFF,
                        "control_after_generate": True,
                    },
                )
            }
        }

    RETURN_TYPES = ("NOISE",)
    FUNCTION = "get_noise"
    CATEGORY = "LTX25/local"

    def get_noise(self, noise_seed):
        return (_LTX25NestedNoise(noise_seed),)


NODE_CLASS_MAPPINGS = {"LTX25NestedRandomNoise": LTX25NestedRandomNoise}
NODE_DISPLAY_NAME_MAPPINGS = {
    "LTX25NestedRandomNoise": "LTX25 Nested Random Noise"
}
'''
(LTX25_NOISE_DIR / "__init__.py").write_text(LTX25_NOISE_CODE, encoding="utf-8")
print("LTX25 nested-noise node: ready")

cmd = [
    sys.executable, "main.py",
    "--listen", "127.0.0.1",
    "--port", str(COMFY_PORT),
    "--enable-cors-header", "*",
    "--cache-none",
    "--async-offload",
    "--reserve-vram", str(COMFY_RESERVE_VRAM_GB),
    "--preview-method", "none",
]
if COMFY_FAST_DISK:
    cmd.append("--fast-disk")
if COMFY_DISABLE_PINNED_MEMORY:
    cmd.append("--disable-pinned-memory")

try:
    if "COMFY_PROC" in globals() and COMFY_PROC.poll() is None:
        COMFY_PROC.terminate()
        COMFY_PROC.wait(timeout=10)
except Exception:
    try:
        COMFY_PROC.kill()
    except Exception:
        pass

log_handle = open(COMFY_LOG, "w", buffering=1)
COMFY_PROC = subprocess.Popen(
    cmd, cwd=COMFY_DIR, stdout=log_handle, stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
COMFY_PID_FILE = Path("/content/comfyui_ltx25.pid")
COMFY_PID_FILE.write_text(str(COMFY_PROC.pid), encoding="utf-8")

print("ComfyUI PID:", COMFY_PROC.pid)
print("ComfyUI log:", COMFY_LOG)
print("Command:", " ".join(cmd))

for _ in range(180):
    if COMFY_PROC.poll() is not None:
        print(COMFY_LOG.read_text(errors="replace")[-16000:])
        raise RuntimeError("ComfyUI exited during startup.")
    try:
        r = requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        if r.ok:
            print("ComfyUI is ready.")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print(COMFY_LOG.read_text(errors="replace")[-16000:])
    raise TimeoutError("ComfyUI did not become ready.")

log_text = COMFY_LOG.read_text(errors="replace")
important_failures = [
    line for line in log_text.splitlines()
    if "IMPORT FAILED" in line.upper() or "FAILED TO IMPORT" in line.upper()
]
if important_failures:
    print("\nCustom-node import warnings found:")
    print("\n".join(important_failures[-30:]))
else:
    print("No obvious custom-node IMPORT FAILED lines in startup log.")

## 6. 圖片生影片（選用）

`MODE="t2v"` 直接跳過上傳；`MODE="i2v"` 則上傳 PNG/JPG 作為影片的第一張圖片。

In [ ]:
# Leave this empty for text-to-video.
# 文字生影片不需要圖片。
IMAGE_PATH = None

# Upload the starting image only for image-to-video.
# 只有圖片生影片才需要上傳起始圖片。
if MODE == "i2v":
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded))
    print("Uploaded:", IMAGE_PATH)

## 7. 直接生成 MP4（等待這格完成）

In [ ]:
# Load the local workflow helper in this notebook process.
# 在 Notebook 程序中載入本機工作流程工具。
import importlib.util
spec = importlib.util.spec_from_file_location("ltx25_engine", "/content/ltx25_engine.py")
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)

# Use placeholders unless image-to-video needs a real starting image.
# 沒有使用起始圖片時，以內部預設圖片完成 Workflow 檢查。
image_name = engine.ensure_placeholder_image()
if MODE == "i2v":
    image_path = Path(IMAGE_PATH)
    image_name, _ = engine.save_upload_bytes(image_path.read_bytes(), image_path.suffix, "lesson")

# Set the controls expected by the original LTX-2.5 workflow.
# 將教學設定填入原始 LTX-2.5 工作流程。
params = dict(mode=MODE, prompt=PROMPT, width=WIDTH, height=HEIGHT,
              seconds=SECONDS, fps=FPS, seed=SEED, image_name=image_name,
              audio_name=engine.ensure_placeholder_audio(), image_strength=IMAGE_STRENGTH,
              vae_tile_size=VAE_TILE_SIZE, vae_tile_overlap=VAE_TILE_OVERLAP,
              vae_temporal_size=VAE_TEMPORAL_SIZE, vae_temporal_overlap=VAE_TEMPORAL_OVERLAP)

# Generate directly and save the MP4 in /content.
# 直接產生影片，並將 MP4 儲存至 /content。
engine.wait_comfy_ready()
result = engine.run_once("lesson", params, MODEL_TIER)
OUTPUT = Path("/content/ltx25_output.mp4")
OUTPUT.write_bytes(Path(result).read_bytes())
print("生成完成：", OUTPUT)

## 8. 預覽並下載 MP4

In [ ]:
# Preview the completed video in Colab.
# 在 Colab 中預覽已完成的影片。
assert OUTPUT.is_file() and OUTPUT.stat().st_size > 0, "尚未產生 MP4。"
display(Video(str(OUTPUT), embed=True))

# Download the MP4 directly to your computer.
# 將 MP4 直接下載至本機電腦。
files.download(str(OUTPUT))

### 記憶體不足（OOM）時

重新啟動 Colab 執行階段，先把 `MODEL_TIER="q2"`、`WIDTH, HEIGHT = 448, 256`，並縮短片長。此做法**只能降低風險，不保證 T4 能完成生成**。若系統 RAM／磁碟不足，GPU 即使有空間也可能失敗。